In [1]:
import pandas as pd 
import numpy as np

import datetime


In [2]:
df_train = pd.read_csv("train.csv", parse_dates=["date"])
df_test = pd.read_csv("test.csv", parse_dates=["date"])


In [3]:
df_train.info()
# print("###################################################")

# df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype         
---  ------       -----         
 0   id           int64         
 1   date         datetime64[us]
 2   store_nbr    int64         
 3   family       str           
 4   sales        float64       
 5   onpromotion  int64         
dtypes: datetime64[us](1), float64(1), int64(3), str(1)
memory usage: 137.4 MB


In [4]:
df_train.sample(5)

,id,date,store_nbr,family,sales,onpromotion
2737844,2737844,2017-03-21,28,SEAFOOD,2.000,0
281328,281328,2013-06-07,52,BEVERAGES,0.000,0
1704390,1704390,2015-08-17,31,CELEBRATION,9.000,0
1030573,1030573,2014-08-03,25,HOME AND KITCHEN II,0.000,0
1836643,1836643,2015-10-30,41,POULTRY,168.621,22


In [5]:
df_train.columns

Index(['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion'], dtype='str')

In [6]:
print(df_train.isna().sum(),'\n','***************')
print(df_train.duplicated().sum())

id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64 
 ***************
0


In [ ]:
# Merge stores, holidays, and oil data
print("Loading external data files...")
df_stores = pd.read_csv("stores.csv")
df_holidays = pd.read_csv("holidays_events.csv", parse_dates=["date"])
df_oil = pd.read_csv("oil.csv", parse_dates=["date"])
print("✓ All files loaded\n")

# Merge stores (on store_nbr)
df_train = df_train.merge(df_stores, on="store_nbr", how="left")
df_test = df_test.merge(df_stores, on="store_nbr", how="left")
print(f"✓ Merged stores:")
print(f"  Train: {df_train.shape}")
print(f"  Test: {df_test.shape}")

# Merge holidays (on date) - use suffixes to avoid duplicate 'type' column
holidays_subset = df_holidays[["date", "type", "locale", "locale_name", "description", "transferred"]]
df_train = df_train.merge(holidays_subset, on="date", how="left", suffixes=("_store", "_holiday"))
df_test = df_test.merge(holidays_subset, on="date", how="left", suffixes=("_store", "_holiday"))
print(f"✓ Merged holidays:")
print(f"  Train: {df_train.shape}")
print(f"  Test: {df_test.shape}")

# Rename holiday type column for clarity, and store type back to "type"
df_train = df_train.rename(columns={"type_holiday": "holiday_type", "type_store": "type"})
df_test = df_test.rename(columns={"type_holiday": "holiday_type", "type_store": "type"})

# Merge oil (on date)
df_train = df_train.merge(df_oil[["date", "dcoilwtico"]], on="date", how="left")
df_test = df_test.merge(df_oil[["date", "dcoilwtico"]], on="date", how="left")
print(f"✓ Merged oil:")
print(f"  Train: {df_train.shape}")
print(f"  Test: {df_test.shape}")

# Fill missing values
for col in ["holiday_type", "locale", "locale_name", "description"]:
    if col in df_train.columns:
        df_train[col] = df_train[col].fillna("No Holiday")
        df_test[col] = df_test[col].fillna("No Holiday")

for col in ["transferred"]:
    if col in df_train.columns:
        df_train[col] = df_train[col].fillna(False)
        df_test[col] = df_test[col].fillna(False)

# Forward fill then backward fill for oil prices
df_train["dcoilwtico"] = df_train["dcoilwtico"].bfill().ffill()
df_test["dcoilwtico"] = df_test["dcoilwtico"].bfill().ffill()

print(f"\n✓ Feature merge complete!")
print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")
print(f"\nTrain columns: {df_train.columns.tolist()}")
print(f"Test columns: {df_test.columns.tolist()}")


Loading external data files...
✓ All files loaded

✓ Merged stores:
  Train: (3000888, 10)
  Test: (28512, 9)
✓ Merged holidays:
  Train: (3054348, 15)
  Test: (28512, 14)
✓ Merged oil:
  Train: (3054348, 16)
  Test: (28512, 15)

✓ Feature merge complete!
Train shape: (3054348, 16)
Test shape: (28512, 15)

Train columns: ['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city', 'state', 'type_store', 'cluster', 'holiday_type', 'locale', 'locale_name', 'description', 'transferred', 'dcoilwtico']
Test columns: ['id', 'date', 'store_nbr', 'family', 'onpromotion', 'city', 'state', 'type_store', 'cluster', 'holiday_type', 'locale', 'locale_name', 'description', 'transferred', 'dcoilwtico']


In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Features to use for training. Keep only columns that exist in both train and test.
feature_columns = [
    'store_nbr',
    'family',
    'onpromotion',
    'type',
    'city',
    'state',
    'cluster',
    'holiday_type',
    'locale',
    'locale_name',
    'description',
    'transferred',
    'dcoilwtico'
]
feature_columns = [c for c in feature_columns if c in df_train.columns and c in df_test.columns]

X_train_full = df_train[feature_columns].copy()
y_train_full = df_train['sales'].copy()
X_test_full = df_test[feature_columns].copy()

# Encode categorical columns consistently between train and test
for col in X_train_full.select_dtypes(include=['object', 'bool']).columns:
    categories = X_train_full[col].astype('category').cat.categories
    X_train_full[col] = X_train_full[col].astype('category').cat.codes
    X_test_full[col] = pd.Categorical(X_test_full[col], categories=categories).codes

# XGBoost regressor with GridSearchCV (internal CV only)
model = XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=1, verbosity=1)
param_grid = {
    'n_estimators': [50], # Reduced for faster run
    'max_depth': [4], 
    'learning_rate': [0.1]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=1,
    refit=True
)

# Convert to numpy arrays to bypass the XGBoost feature names bug with Pandas
grid.fit(X_train_full.values, y_train_full.values)

best_model = grid.best_estimator_
print('Best params:', grid.best_params_)
print('Best CV score (negative RMSE):', grid.best_score_)

# The refitted best model is trained on the full training set using numpy arrays
train_pred = best_model.predict(X_train_full.values)
rmse_train = np.sqrt(mean_squared_error(y_train_full, train_pred))
mae_train = mean_absolute_error(y_train_full, train_pred)
r2_train = r2_score(y_train_full, train_pred)
print(f'Train RMSE: {rmse_train:.4f}')
print(f'Train MAE: {mae_train:.4f}')
print(f'Train R2: {r2_train:.4f}')

# Predict on test set
df_test['predicted_sales'] = best_model.predict(X_test_full.values)
print('\nPrediction completed on df_test')
print(df_test[['id', 'predicted_sales']].head())

In [ ]:
import joblib
joblib.dump(best_model, "model.pkl")

['model.pkl']